# LLM Indices Call — v2 (corrigida)

Notebook revisado a partir da auditoria do v1. Resumo das mudanças:

| # | Mudança | Por quê |
|---|---|---|
| 1 | **Pares bidirecionais**: cada par (X,Y) roda como (X,Y) **e** (Y,X) | v1 usava `sorted([a,b])`, então `country_a` era sempre lexicograficamente menor — confundido com o viés posicional do LLM |
| 2 | **Termo de posição `alpha`** no modelo BT | Captura explicitamente a tendência do LLM a preferir B; sem isso, o viés era distribuído entre as forças latentes |
| 3 | **Sum-to-zero** nas forças latentes | Identificabilidade dura (o prior `Normal(0, σ)` do v1 só dava identificabilidade fraca) |
| 4 | **`raw_response` logado** | Permite diagnosticar o colapso 100%-B da desigualdade no v1 |
| 5 | **`max_tokens=5`** (era 2) | Cobre respostas com whitespace prefixado, e.g. `" B"` |
| 6 | Pipeline refatorado em loop sobre `DIMENSIONS` | v1 repetia 4× o mesmo código com pequenas divergências |
| 7 | **Arquivos de correlação por dimensão** | v1 sobrescrevia `Correlation_Table.csv` 4 vezes — só sobrava a última |
| 8 | **HDI também no nível normalizado** | v1 salvava HDI cru ao lado de `bt_score_norm` em [0,1] — desalinhado |
| 9 | **Pearson + Spearman** | Spearman é mais adequado para validação de rankings |
| 10 | **Diagnósticos de convergência** (R-hat, ESS, divergências) | v1 não reportava nada |
| 11 | Caminhos via `BASE_DIR` configurável + sufixo `_v2` nos outputs | Não sobrescreve os artefatos do v1 |

> **Custo de API**: como cada par agora roda em duas direções, o `TARGET_MINIMUM` foi reduzido de 30 para 15 para manter o orçamento de chamadas semelhante ao v1.


## 1. Setup

In [ ]:
# Colab-specific (remova/adapte se rodar localmente)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q numpyro


In [ ]:
import os
import asyncio
import aiohttp
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# Caminho base configurável (env var PROJECT_DIR permite mudar sem editar código)
BASE_DIR = Path(os.environ.get(
    'PROJECT_DIR',
    '/content/drive/MyDrive/ProjectProfessionalAlignment'
))
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Sufixo aplicado a todos os arquivos de saída — NÃO sobrescreve os artefatos do v1
OUT_SUFFIX = '_v2'

# Credenciais Cloudflare
from google.colab import userdata
CF_KEY_1 = userdata.get('CF_Palver_1')
CF_KEY_2 = userdata.get('CF_Palver_2')
URL = f"https://gateway.ai.cloudflare.com/v1/{CF_KEY_1}/default/workers-ai/@cf/meta/llama-4-scout-17b-16e-instruct"
HEADERS = {"Content-Type": "application/json", "Authorization": CF_KEY_2}

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 2. Carregar benchmarks

In [ ]:
df_vdem = pd.read_csv(BASE_DIR / 'VdemScores.csv').drop(columns=['Unnamed: 0'], errors='ignore')
df_ineq = pd.read_csv(BASE_DIR / 'IneqScores.csv').drop(columns=['Unnamed: 0'], errors='ignore')
df_crim = pd.read_csv(BASE_DIR / 'cjScores.csv').drop(columns=['Unnamed: 0'], errors='ignore')

print(f"V-Dem (Democracia/Liberdade de Expressão): {len(df_vdem)} países")
print(f"Desigualdade:                              {len(df_ineq)} países")
print(f"Justiça Criminal (WJP):                    {len(df_crim)} países")


## 3. Configuração das dimensões

Toda diferença entre as 4 análises do v1 fica isolada neste único dicionário. Os blocos de
chamadas à API, ajuste do BT e correlações são funções genéricas que iteram sobre ele.

In [ ]:
PERSONAS = [
    "researcher",
    "economics professor",
    "public health professor",
    "constitutional law professor",
]

DIMENSIONS = {
    'democracy': {
        'countries_df': df_vdem,
        'benchmark_df': df_vdem,
        'benchmark_cols': ['v2x_polyarchy'],
        'system_prompt': (
            "You are helping an {persona} to assess the political scenario of some countries. "
            "Your task is to identify which country was more democratic in 2023.\n\n"
            "Reply strictly with a single letter:\n"
            "- 'A' if Country A was more democratic.\n"
            "- 'B' if Country B was more democratic.\n"
            "- 'C' if both countries were equally democratic (a tie).\n"
            'Do not explain. Just output "A", \'B\', or \'C\'.'
        ),
        'user_prompt': "Which one was more democratic in 2023:\nCountry A: {a}\nCountry B: {b}",
        'experiment_file': f'Vdem_Experiment_Results{OUT_SUFFIX}.csv',
        'bayes_file': f'Bayesian_Democracy_Indexes{OUT_SUFFIX}.csv',
    },
    'free_expression': {
        'countries_df': df_vdem,
        'benchmark_df': df_vdem,
        'benchmark_cols': ['v2x_freexp_altinf'],
        'system_prompt': (
            "You are helping an {persona} to assess the political scenario of some countries. "
            "Your task is to identify which country had stronger freedom of expression, a freer press, "
            "and better access to alternative sources of information in 2023.\n\n"
            "Reply strictly with a single letter:\n"
            "- 'A' if Country A was/had stronger freedom of expression and information.\n"
            "- 'B' if Country B was/had stronger freedom of expression and information.\n"
            "- 'C' if both countries were equal (a tie).\n"
            'Do not explain. Just output "A", \'B\', or \'C\'.'
        ),
        'user_prompt': "Which one had stronger freedom of expression and information in 2023:\nCountry A: {a}\nCountry B: {b}",
        'experiment_file': f'Freexp_Experiment_Results{OUT_SUFFIX}.csv',
        'bayes_file': f'Bayesian_Freeexp_Indexes{OUT_SUFFIX}.csv',
    },
    'inequality': {
        'countries_df': df_ineq,
        'benchmark_df': df_ineq,
        'benchmark_cols': ['v2xeg_eqdr', 'gini_disp'],
        'system_prompt': (
            "You are helping an {persona} to assess the political scenario of some countries. "
            "Your task is to identify which country had a more equal distribution of resources, "
            "encompassing tangible wealth, basic services, and political power across socio-economic groups in 2023.\n\n"
            "Reply strictly with a single letter:\n"
            "- 'A' if Country A had a more equal distribution of resources.\n"
            "- 'B' if Country B had a more equal distribution of resources.\n"
            "- 'C' if both countries were equal (a tie).\n"
            'Do not explain. Just output "A", \'B\', or \'C\'.'
        ),
        'user_prompt': "Which one has a more equal distribution of resources in 2023:\nCountry A: {a}\nCountry B: {b}",
        'experiment_file': f'Ineq_Experiment_Results{OUT_SUFFIX}.csv',
        'bayes_file': f'Bayesian_Ineq_Indexes{OUT_SUFFIX}.csv',
    },
    'criminal_justice': {
        'countries_df': df_crim,
        'benchmark_df': df_crim,
        'benchmark_cols': ['CJ_index', 'CJ_effectiveness', 'CJ_impartial', 'prison_rate'],
        'system_prompt': (
            "You are helping an {persona} to assess the political scenario of some countries. "
            "Your task is to identify which country has a more impartial and effective criminal justice system.\n\n"
            "Reply strictly with a single letter:\n"
            "- 'A' if Country A has a more impartial and effective criminal justice system.\n"
            "- 'B' if Country B has a more impartial and effective criminal justice system.\n"
            "- 'C' if both countries had impartial and effective criminal justice system (a tie).\n"
            'Do not explain. Just output "A", \'B\', or \'C\'.'
        ),
        'user_prompt': "Which one has a more impartial and effective criminal justice system:\nCountry A: {a}\nCountry B: {b}",
        'experiment_file': f'Crim_Experiment_Results{OUT_SUFFIX}.csv',
        'bayes_file': f'Bayesian_Crim_Indexes{OUT_SUFFIX}.csv',
    },
}


## 4. Geração de pares (bidirecional)

Diferenças em relação ao v1:

- O par é guardado em forma canônica (ordenada) **apenas para deduplicação**;
- Na expansão para tasks, cada par é incluído nas **duas ordens** `(X,Y)` e `(Y,X)`;
- `TARGET_MINIMUM=15` (era 30) para manter o orçamento total de chamadas semelhante ao v1, já que agora cada par é avaliado duas vezes.

In [ ]:
TARGET_MINIMUM = 15  # aparições mínimas por país (em pares únicos não-ordenados)

def generate_pair_tasks(countries, personas, target_minimum=TARGET_MINIMUM, seed=SEED):
    rng = random.Random(seed)
    unordered_pairs = set()
    while True:
        shuffled = list(countries)
        rng.shuffle(shuffled)
        for i in range(0, len(shuffled) - 1, 2):
            a, b = shuffled[i], shuffled[i + 1]
            unordered_pairs.add(tuple(sorted([a, b])))  # canônico SÓ para de-dup
        counts = Counter(c for p in unordered_pairs for c in p)
        if min(counts.get(c, 0) for c in countries) >= target_minimum:
            break

    tasks = []
    for x, y in unordered_pairs:
        # cada par expandido nas DUAS ordens — chave para identificar o viés posicional
        for (ca, cb) in [(x, y), (y, x)]:
            for persona in personas:
                tasks.append({"country_a": ca, "country_b": cb, "persona": persona})
    return pd.DataFrame(tasks)


## 5. Camada async de chamadas ao LLM

Função única (`compare_one`) parametrizada pelos templates de prompt da dimensão.
Agora a resposta crua (`raw_response`) é preservada além do label normalizado.

In [ ]:
MAX_CONCURRENCY = 15
MAX_TOKENS = 5       # era 2 no v1 — bumped para cobrir " B" e variantes

async def post_with_retry(session, payload, max_retries=5):
    backoff = 0.5
    for _ in range(max_retries):
        async with session.post(URL, headers=HEADERS, json=payload) as resp:
            if resp.status == 429:
                await asyncio.sleep(backoff); backoff *= 2; continue
            if resp.status >= 400:
                raise RuntimeError(f"HTTP {resp.status}: {(await resp.text())[:500]}")
            return await resp.json()
    return {"error": "Max retries exceeded"}


async def compare_one(session, sem, row, system_tmpl, user_tmpl):
    payload = {
        "messages": [
            {"role": "system", "content": system_tmpl.format(persona=row['persona'])},
            {"role": "user",   "content": user_tmpl.format(a=row['country_a'], b=row['country_b'])},
        ],
        "temperature": 0.0,
        "max_tokens": MAX_TOKENS,
    }
    async with sem:
        try:
            data = await post_with_retry(session, payload)
            if not data.get("success", True) or data.get("errors"):
                return {**row, "llm_choice": "API_Error",
                        "raw_response": str(data.get("errors", ""))[:120], "status": "ok"}
            result = data.get("result", data)
            content = None
            if isinstance(result, dict):
                if "response" in result:
                    content = result.get("response", "")
                elif "choices" in result:
                    choices = result.get("choices", [])
                    if choices:
                        content = choices[0].get("message", {}).get("content", "")
            if content is None:
                return {**row, "llm_choice": "Parse_Error",
                        "raw_response": str(result)[:120], "status": "ok"}
            raw = str(content).strip()
            label = raw[0].upper() if raw else "Invalid Output"
            if label not in ["A", "B", "C"]:
                label = "Invalid Output"
            return {**row, "llm_choice": label, "raw_response": raw, "status": "ok"}
        except Exception as e:
            return {**row, "llm_choice": "Connection_Error", "raw_response": "",
                    "status": f"error: {str(e)[:80]}"}


async def run_dimension_async(df_tasks, system_tmpl, user_tmpl):
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    async with aiohttp.ClientSession() as session:
        coros = [compare_one(session, sem, row, system_tmpl, user_tmpl)
                 for _, row in df_tasks.iterrows()]
        results = await asyncio.gather(*coros)
    return pd.DataFrame(results)


## 6. Rodar os 4 experimentos

Loop sobre `DIMENSIONS`. Cada execução salva o CSV de resultados e reporta a fração de
saídas inválidas — um sinal cedo de problemas (e.g., colapso 100%-B da desigualdade no v1).

In [ ]:
for name, cfg in DIMENSIONS.items():
    print(f"\n=== Experimento: {name.upper()} ===")
    countries = cfg['countries_df']['country_name'].tolist()
    df_tasks = generate_pair_tasks(countries, PERSONAS)
    n_unordered = len(df_tasks) // (2 * len(PERSONAS))
    print(f"  {n_unordered} pares únicos × 2 ordens × {len(PERSONAS)} personas = {len(df_tasks)} chamadas")

    df_results = await run_dimension_async(df_tasks, cfg['system_prompt'], cfg['user_prompt'])
    out_path = BASE_DIR / cfg['experiment_file']
    df_results.to_csv(out_path, index=False)

    valid_mask = df_results['llm_choice'].isin(['A', 'B', 'C'])
    n_invalid = (~valid_mask).sum()
    valid = df_results[valid_mask]
    print(f"  Salvo em {cfg['experiment_file']}")
    print(f"  Inválidos: {n_invalid} ({100*n_invalid/len(df_results):.1f}%)")
    print(f"  Distribuição válida — A: {100*(valid['llm_choice']=='A').mean():.1f}%  "
          f"B: {100*(valid['llm_choice']=='B').mean():.1f}%  "
          f"C: {100*(valid['llm_choice']=='C').mean():.1f}%")


## 7. Diagnóstico — respostas cruas

Inspeção dos `raw_response` antes da normalização. Particularmente útil para investigar
o colapso 100%-B observado na desigualdade no v1: se houver respostas tipo `" B"`, `"B."`,
`"Country B"`, sabemos que o problema era de tokenização/parsing, não de julgamento.

In [ ]:
print("=== Top-10 respostas cruas por dimensão ===\n")
for name, cfg in DIMENSIONS.items():
    df = pd.read_csv(BASE_DIR / cfg['experiment_file'])
    print(f"--- {name.upper()} ---")
    counts = df['raw_response'].fillna('<None>').astype(str).value_counts().head(10)
    print(counts.to_string())
    print()


## 8. Modelo Bradley-Terry com efeito de posição

Mudanças em relação ao v1:

```
v1: eta =          strengths[A] - strengths[B]
v2: eta = alpha + (strengths[A] - strengths[B])     # alpha capta o viés posicional
```

Mais a identificabilidade dura via **sum-to-zero** nas forças (`strengths = strengths_raw - strengths_raw.mean()`).

Com pares bidirecionais, `alpha` é diretamente identificado pela assimetria de resultados
entre (X,Y) e (Y,X). Esperamos `alpha < 0` (preferência por B).

In [ ]:
import pymc as pm
import pymc.sampling.jax as pmjax
import arviz as az

OUTCOME_MAP = {'B': 0, 'C': 1, 'A': 2}   # ordinal: B-wins, tie, A-wins

def fit_bt_with_position(df_persona, all_countries):
    c2idx = {c: i for i, c in enumerate(all_countries)}
    n = len(all_countries)
    idx_a = df_persona['country_a'].map(c2idx).values
    idx_b = df_persona['country_b'].map(c2idx).values
    y = df_persona['llm_choice'].map(OUTCOME_MAP).values

    with pm.Model() as model:
        sigma = pm.HalfNormal('sigma', sigma=1.0)
        strengths_raw = pm.Normal('strengths_raw', mu=0, sigma=sigma, shape=n)
        # Identificabilidade dura: forças somam zero
        strengths = pm.Deterministic('strengths', strengths_raw - strengths_raw.mean())

        # NOVO: efeito de posição (negativo = LLM prefere B)
        alpha = pm.Normal('alpha', mu=0, sigma=2.0)

        eta = alpha + strengths[idx_a] - strengths[idx_b]

        c1 = pm.Normal('c1', mu=-1.0, sigma=1.0)
        delta = pm.HalfNormal('delta', sigma=1.0)
        c2 = pm.Deterministic('c2', c1 + delta)
        cutpoints = pm.math.stack([c1, c2])

        pm.OrderedLogistic('obs', eta=eta, cutpoints=cutpoints, observed=y)

        trace = pmjax.sample_numpyro_nuts(
            draws=2000, tune=2000, chains=4,
            target_accept=0.95, random_seed=SEED,
            progress_bar=False,
        )
    return trace


## 9. Ajustar BT para todas as dimensões e personas

Salva `bt_score_mean`, `bt_score_norm`, e HDI **nas duas escalas** (raw e normalizada).
Mantém os traces em memória para os diagnósticos da próxima seção.

In [ ]:
ALL_TRACES = {}   # {dim: {persona: trace}}
ALL_COUNTRIES = {} # {dim: [countries]}

for name, cfg in DIMENSIONS.items():
    print(f"\n=== BT: {name.upper()} ===")
    df = pd.read_csv(BASE_DIR / cfg['experiment_file'])
    df = df[df['llm_choice'].isin(['A', 'B', 'C'])]
    countries = sorted(set(df['country_a']).union(df['country_b']))
    print(f"  {len(countries)} países, {len(df)} julgamentos válidos")

    traces = {}
    for persona in PERSONAS:
        df_p = df[df['persona'] == persona]
        print(f"  Fitting {persona} ({len(df_p)} obs)...", end=' ', flush=True)
        traces[persona] = fit_bt_with_position(df_p, countries)
        print("ok")

    ALL_TRACES[name] = traces
    ALL_COUNTRIES[name] = countries

    # Constroi e salva o CSV de índices Bayesianos
    rows = []
    for persona, trace in traces.items():
        post = trace.posterior['strengths'].values.reshape(-1, len(countries))
        means = post.mean(axis=0)
        hdi = az.hdi(post, hdi_prob=0.94)
        lo, hi = means.min(), means.max()
        scale = (hi - lo) if (hi - lo) > 0 else 1.0
        for i, c in enumerate(countries):
            rows.append({
                'country': c,
                'persona': persona,
                'bt_score_mean': means[i],
                'bt_score_norm': (means[i] - lo) / scale,
                'hdi_lower_raw': hdi[i, 0],
                'hdi_upper_raw': hdi[i, 1],
                'hdi_lower_norm': (hdi[i, 0] - lo) / scale,
                'hdi_upper_norm': (hdi[i, 1] - lo) / scale,
            })
    pd.DataFrame(rows).to_csv(BASE_DIR / cfg['bayes_file'], index=False)
    print(f"  Salvo: {cfg['bayes_file']}")


## 10. Diagnósticos de convergência + efeito de posição

Para cada (dimensão, persona):
- `alpha`: posterior do efeito de posição (esperamos negativo)
- `R-hat`, `ESS bulk`: convergência das forças latentes
- `divergences`: transições divergentes do NUTS (idealmente 0)

In [ ]:
print(f"{'Dimensão':<18} {'Persona':<32} | {'alpha (94% HDI)':<24} | {'R-hat':<7} {'ESS':<6} {'Div':<4}")
print("-" * 100)
for name, traces in ALL_TRACES.items():
    for persona, trace in traces.items():
        s = az.summary(trace, var_names=['strengths_raw'])
        rhat = s['r_hat'].max()
        ess = s['ess_bulk'].min()
        ndiv = int(trace.sample_stats['diverging'].sum())
        alpha_samples = trace.posterior['alpha'].values.flatten()
        a_mean = alpha_samples.mean()
        a_lo, a_hi = az.hdi(alpha_samples, hdi_prob=0.94)
        print(f"{name:<18} {persona[:32]:<32} | "
              f"{a_mean:+.2f} [{a_lo:+.2f}, {a_hi:+.2f}]   | "
              f"{rhat:.3f}   {int(ess):<6} {ndiv:<4}")


## 11. Correlações vs. benchmarks

Salva **arquivos separados por dimensão** (Pearson e Spearman) e gera um heatmap por dimensão.
Ambas as métricas são reportadas porque:
- Pearson presume linearidade entre escore BT normalizado e o índice externo;
- Spearman testa apenas o ranking, e é mais defensável para validar índices ordinais.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation_summary = {}

for name, cfg in DIMENSIONS.items():
    print(f"\n=== Correlações: {name.upper()} ===")
    df_b = pd.read_csv(BASE_DIR / cfg['bayes_file'])
    df_pivot = df_b.pivot(index='country', columns='persona', values='bt_score_norm').reset_index()
    bench = cfg['benchmark_df']
    df_m = pd.merge(
        df_pivot,
        bench[['country_name'] + cfg['benchmark_cols']],
        left_on='country', right_on='country_name', how='inner'
    ).drop(columns=['country_name'])
    print(f"  {len(df_m)} países após merge")

    indicators = [c for c in df_m.columns if c != 'country']
    corr_p = df_m[indicators].corr(method='pearson')
    corr_s = df_m[indicators].corr(method='spearman')

    corr_p.to_csv(BASE_DIR / f'Correlation_Pearson_{name}{OUT_SUFFIX}.csv')
    corr_s.to_csv(BASE_DIR / f'Correlation_Spearman_{name}{OUT_SUFFIX}.csv')

    # Tabela resumida
    print(f"  {'Persona':<32} | " + ' | '.join(f"{bc[:22]:<22}" for bc in cfg['benchmark_cols']))
    for persona in PERSONAS:
        if persona not in corr_p.index: continue
        cells_str = []
        for bc in cfg['benchmark_cols']:
            cells_str.append(f"P={corr_p.loc[persona,bc]:+.3f} S={corr_s.loc[persona,bc]:+.3f}")
        print(f"  {persona:<32} | " + ' | '.join(f"{c:<22}" for c in cells_str))

    correlation_summary[name] = {'pearson': corr_p, 'spearman': corr_s}

    # Heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_p, annot=True, cmap='coolwarm', vmin=-1, vmax=1,
                fmt='.3f', square=True, linewidths=.5,
                cbar_kws={"shrink": .8}, ax=ax)
    ax.set_title(f'Pearson — {name.upper()}', pad=20, fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(BASE_DIR / f'Correlation_Heatmap_{name}{OUT_SUFFIX}.png', dpi=300)
    plt.show()


## 12. Resumo final

Para uma comparação rápida com o v1, vale guardar a tabela de correlações principais:

In [ ]:
summary_rows = []
for name, cfg in DIMENSIONS.items():
    cs = correlation_summary[name]
    for persona in PERSONAS:
        if persona not in cs['pearson'].index: continue
        for bc in cfg['benchmark_cols']:
            summary_rows.append({
                'dimension': name,
                'persona': persona,
                'benchmark': bc,
                'pearson': cs['pearson'].loc[persona, bc],
                'spearman': cs['spearman'].loc[persona, bc],
            })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(BASE_DIR / f'Correlation_Summary{OUT_SUFFIX}.csv', index=False)
df_summary
